# Diabetes Readmission — Outreach Prioritization

## Project overview

This project explores whether machine learning can help prioritize
post-discharge follow-up for hospital encounters involving patients
with diabetes.

The operational question is:
Which discharge encounters should receive priority when outreach
capacity is limited to 10% of eligible discharges?

The workflow compares logistic regression and random forest models,
ranks discharge encounters by model score, and exports the results
for a Power BI dashboard.

This is a retrospective research prototype, not a clinical tool.
No outreach was performed, and reduced readmissions or financial
savings were not measured.

## 1. Dataset and unit of analysis

- Dataset: UCI Diabetes 130-US Hospitals, 1999–2008.
- Original size: 101,766 hospital encounters.
- Unit of analysis: one hospital encounter, not one unique patient.
- Patients may have multiple encounters.
- Outcome: recorded readmission within 30 days.

The dataset includes demographics, admission and discharge
information, diagnoses, medication information, and prior
healthcare utilization.

Important limitations:
- Historical US data may not represent current practice or other countries.
- The selected encounters involve patients with diabetes.
- The analysis predicts the recorded outcome; it does not establish
  whether a readmission was preventable.

## 2. Imports and setup


In [3]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score


## 3. Load the source files

In [4]:

df = pd.read_csv('diabetic_data.csv', low_memory=False)
rows = []
for line in open("IDS_mapping.csv", encoding="utf-8"):
    line = line.strip()
    if "," not in line:
        continue
    code, description = line.split(",", 1)
    rows.append((code.strip().strip('"'), description.strip().strip('"')))

ids = pd.DataFrame(rows, columns=['code', 'description'])
ids.head(20)


,code,description
0,admission_type_id,description
1,1,Emergency
2,2,Urgent
3,3,Elective
4,4,Newborn
5,5,Not Available
6,6,NULL
7,7,Trauma Center
8,8,Not Mapped
9,,


## 4. Inspect the original data

In [5]:
aux = df.isna().sum(axis=1)
print("\nMissing values per row:")
print(aux.head())

print("Outcome counts:")
print(df["readmitted"].value_counts())

print("\nOutcome proportions:")
print(df["readmitted"].value_counts(normalize=True).round(3))






Missing values per row:
0    2
1    2
2    2
3    2
4    2
dtype: int64
Outcome counts:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Outcome proportions:
readmitted
NO     0.539
>30    0.349
<30    0.112
Name: proportion, dtype: float64


In [6]:
missing = (df == "?").sum().sort_values(ascending=False)
missing[missing > 0]

weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

In [7]:

print("encounter_id", df['encounter_id'].nunique())
print("patient_nbr", df['patient_nbr'].nunique())
print(df["patient_nbr"].duplicated().sum())

encounter_id 101766
patient_nbr 71518
30248


## 5. Define eligibility and the target

In [8]:
start = ids.index[ids["code"] == "discharge_disposition_id"][0] +1 
end = ids.index[ids["code"] == "admission_source_id"][0]

discharge = ids.iloc[start:end].copy()
discharge = discharge[discharge["code"].str.isdigit()]
discharge["code"] = discharge["code"].astype(int)

mask = discharge["description"].str.contains("Expired|Hospice", case=False, na=False)

drop_ids = discharge.loc[mask, "code"].tolist()

print(discharge.loc[mask])
print(drop_ids)

    code                                        description
21    11                                            Expired
23    13                                     Hospice / home
24    14                         Hospice / medical facility
29    19           Expired at home. Medicaid only, hospice.
30    20  Expired in a medical facility. Medicaid only, ...
31    21    Expired, place unknown. Medicaid only, hospice.
[11, 13, 14, 19, 20, 21]


In [9]:
work = df.copy()
work = work[~work["discharge_disposition_id"].isin(drop_ids)]
work = work[work["gender"] != "Unknown/Invalid"]
work["readmit_30"] = (work["readmitted"] == "<30").astype(int)

print(work.shape)
print(work["readmit_30"].mean().round(3))

(99340, 51)
0.114


In [10]:
work[["encounter_id", "patient_nbr", "readmitted", "readmit_30"]].head(20)

,encounter_id,patient_nbr,readmitted,readmit_30
0,2278392,8222157,NO,0
1,149190,55629189,>30,0
2,64410,86047875,NO,0
3,500364,82442376,NO,0
4,16680,42519267,NO,0
5,35754,82637451,>30,0
6,55842,84259809,NO,0
7,63768,114882984,>30,0
8,12522,48330783,NO,0
9,15738,63555939,NO,0


## 6. Apply fixed data-preparation rules

In [11]:
clean = work.copy()
clean = clean.replace("?", pd.NA)

print(clean.isna().mean().sort_values(ascending=False).head(10).round(3))

clean = clean.drop(columns=["weight"])

weight               0.969
max_glu_serum        0.948
A1Cresult            0.831
medical_specialty    0.489
payer_code           0.397
race                 0.022
diag_3               0.014
diag_2               0.004
diag_1               0.000
patient_nbr          0.000
dtype: float64


### 6.1. Create a simplified diagnosis group

In [12]:
def diagnosis_group(code):
    if pd.isna(code):
        return "Other"

    text = str(code)

    if text.startswith("250"):
        return "Diabetes"

    try:
        number = float(text)
    except ValueError:
        return "Other"

    if 390 <= number < 460:
        return "Circulatory"

    return "Other"


clean["diag_group"] = clean["diag_1"].apply(diagnosis_group)

print(clean["diag_group"].value_counts())


clean.to_csv("encounters_clean.csv", index=False)

check = pd.read_csv("encounters_clean.csv", dtype={"payer_code": "string"}, low_memory=False,)
print("Saved table shape:", check.shape)
print("Positive outcome share:", check["readmit_30"].mean().round(3))
print("Diagnosis group present:", "diag_group" in check.columns)

diag_group
Other          61095
Circulatory    29584
Diabetes        8661
Name: count, dtype: int64
Saved table shape: (99340, 51)
Positive outcome share: 0.114
Diagnosis group present: True


## 7. Split the data by patient

In [13]:
clean = pd.read_csv(
    "encounters_clean.csv",
    dtype={"payer_code": "string"},
    low_memory=False,
)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(clean, groups=clean["patient_nbr"]))
train = clean.iloc[train_idx].copy()
test = clean.iloc[test_idx].copy()

train_patients = set(train["patient_nbr"])
test_patients = set(test["patient_nbr"])
overlap = train_patients & test_patients


print("train stays:", len(train))
print("test stays:", len(test))
print("train people:", len(train_patients))
print("test people:", len(test_patients))
print("people in both:", len(overlap))
print("train positive share:", train["readmit_30"].mean().round(3))
print("test positive share:", test["readmit_30"].mean().round(3))

train stays: 79567
test stays: 19773
train people: 55989
test people: 13998
people in both: 0
train positive share: 0.114
test positive share: 0.113


### 7.1. Group rare categories using training data only

Category frequencies are calculated from the training set.
The same category rule is then applied to both training and test data.

In [14]:
category_columns = ["payer_code", "medical_specialty"]

for column in category_columns:
    counts = train[column].value_counts()
    common_categories = counts[counts >= 1000].index

    train[column] = train[column].where(
        train[column].isin(common_categories),
        other="Other",
    )

    test[column] = test[column].where(
        test[column].isin(common_categories),
        other="Other",
    )

    print(f"\n{column}:")
    print(train[column].value_counts())


payer_code:
payer_code
Other    33693
MC       25054
HM        5010
SP        3941
BC        3679
MD        2788
CP        1949
UN        1947
CM        1506
Name: count, dtype: Int64

medical_specialty:
medical_specialty
Other                     47372
InternalMedicine          11453
Emergency/Trauma           5926
Family/GeneralPractice     5790
Cardiology                 4235
Surgery-General            2488
Nephrology                 1204
Orthopedics                1099
Name: count, dtype: int64


## 8. Select model inputs and target


In [15]:
y_train = train["readmit_30"]
y_test = test["readmit_30"]

features = [
    "age", "gender", "race", "diag_group",
    "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient",
    "admission_type_id", "discharge_disposition_id",
    "insulin", "change", "diabetesMed", "A1Cresult",
    "payer_code", "medical_specialty",
]

X_train = train[features].copy()
X_test = test[features].copy()


print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(79567, 19) (19773, 19)
0.114 0.113


## 9. Build the preprocessing pipeline

In [16]:
numeric = [
    "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient",
]
categorical = [c for c in features if c not in numeric]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("fill", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])


## 10. Train and evaluate logistic regression

In [17]:
model = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)


proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)


print("precision:", round(precision_score(y_test, pred), 3))
print("recall:", round(recall_score(y_test, pred), 3))
print("roc_auc:", round(roc_auc_score(y_test, proba), 3))

precision: 0.185
recall: 0.535
roc_auc: 0.665


## 11. Train and evaluate the random forest

In [18]:
forest = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

forest.fit(X_train, y_train)


forest_proba = forest.predict_proba(X_test)[:, 1]
forest_pred = (forest_proba >= 0.5).astype(int)


print("logistic recall:", round(recall_score(y_test, pred), 3))
print("forest recall:", round(recall_score(y_test, forest_pred), 3))
print("logistic precision:", round(precision_score(y_test, pred), 3))
print("forest precision:", round(precision_score(y_test, forest_pred), 3))
print("logistic roc_auc:", round(roc_auc_score(y_test, proba), 3))
print("forest roc_auc:", round(roc_auc_score(y_test, forest_proba), 3))

logistic recall: 0.535
forest recall: 0.607
logistic precision: 0.185
forest precision: 0.178
logistic roc_auc: 0.665
forest roc_auc: 0.67


## 12. Evaluate top-10% outreach prioritization

In [19]:
scored = test.copy()
scored["risk"] = proba
scored = scored.sort_values("risk", ascending=False)

n_calls = int(len(scored) * 0.10)
top = scored.head(n_calls)

caught = int(top["readmit_30"].sum())
total_positives = int(scored["readmit_30"].sum())
selection_fraction = n_calls / len(scored)
random_caught = total_positives * selection_fraction
precision_at_capacity = caught / n_calls
recall_at_capacity = caught / total_positives
lift_at_capacity = caught / random_caught

print("Priority encounters:", n_calls)
print("Observed readmissions in priority group:", caught)
print("Total observed test readmissions:", total_positives)
print("Expected under random selection:", round(random_caught, 1))
print("Priority-group readmission rate:", round(precision_at_capacity, 3))
print("Share of readmissions identified:", round(recall_at_capacity, 3))
print("Lift versus random:", round(lift_at_capacity, 2))


Priority encounters: 1977
Observed readmissions in priority group: 516
Total observed test readmissions: 2244
Expected under random selection: 224.4
Priority-group readmission rate: 0.261
Share of readmissions identified: 0.23
Lift versus random: 2.3


## 13. Export results for Power BI

In [20]:
scored["call_list"] = scored.index.isin(top.index)
scored["risk_tier"] = "Rest"
scored.loc[scored.index.isin(scored.head(int(len(scored) * 0.30)).index), "risk_tier"] = "Next 20%"
scored.loc[scored["call_list"], "risk_tier"] = "Top 10%"

demo = scored[["age", "gender", "diag_group", "time_in_hospital",
               "number_inpatient", "risk", "risk_tier", "readmit_30"]].copy()
demo.to_csv("call_list_demo.csv", index=False)
print(demo["risk_tier"].value_counts())


risk_tier
Rest        13842
Next 20%     3954
Top 10%      1977
Name: count, dtype: int64


## 14. Findings, limitations, and next steps

### Findings

The eligible dataset contained 99,340 hospital encounters.
The patient-grouped split produced 79,567 training encounters
and 19,773 test encounters, with no patients shared between the sets.

Logistic regression achieved a test ROC-AUC of 0.665.
Random forest achieved a test ROC-AUC of 0.670.
Logistic regression was retained as the simpler baseline for
the outreach-prioritization demonstration.

Prioritizing the highest-scoring 10% of test encounters selected
1,977 discharge encounters. This group contained 516 of the
2,244 observed 30-day readmissions.

The priority group therefore had:
- A 26.1% observed readmission rate.
- Approximately 23.0% of all test-set readmissions.
- Approximately 2.30 times the expected yield of random selection.

These results describe retrospective identification of readmissions,
not evidence that outreach would prevent them.


### Limitations

- The dataset is historical and drawn from US hospitals.
- The analysis prioritizes encounters, not necessarily unique patients.
- Model scores have not been assessed for probability calibration.
- No outreach intervention was performed.
- Reduced readmissions and financial savings were not measured.
- Model comparison used the test set, so results should be treated as
  exploratory rather than a fully independent final evaluation.
- Performance differences across demographic groups have not been assessed.

### Next steps

- Compare models using patient-grouped validation within training data.
- Keep preprocessing inside that validation workflow.
- Assess probability calibration if probabilities will be presented.
- Assess subgroup performance before considering practical use.

This project is a research and portfolio prototype, not a clinical tool.